# Lab 9 Workbook: Letting the Model Make Its Own Features
## MPS311/439 Machine Learning

The same tasks as the [Lab 9 worksheet](https://ml.wxing.me/materials/lesson9/lab.html), with the code already in place: **nothing to copy**.

**What you do today.** The Python is given. What is left to you are the machine-learning decisions: the blanks `____` and the questions. That is the lab. A cell with a blank stops with an error until you fill it.

After the Parts come the after-class part and the **Advanced Study** (MPS439: investigation A is required). Only the folded Answers are not here: they stay on the [worksheet](https://ml.wxing.me/materials/lesson9/lab.html).

## What this lab is for

In the lecture a few logistic units drew lines around the inner ring, and a network trained them all together. Today you first train that small network on the ring yourself, then the network of Assignment 2, Section 4, on MNIST:

1. train networks with 1 to 4 hidden units on the ring and see the lines they draw;
2. build the MNIST network in Keras and train it for 5 epochs;
3. remove the activation in the middle and see what is lost;
4. draw the training and validation curves and find where the network starts to memorise;
5. read how much the result moves when only the random seed changes.

Before each step you will see a question. **Write down your own answer first**, then run the code and compare.

Most code is provided. Your job is to fill the blanks `____`, run the cell, and check your result against the **Success check**. Training takes time: each training cell says roughly how long it runs in Colab. At the end there is a short section on how today's lab helps with **Assignment 2**.

## How to work

1. **File → Save a copy in Drive**, and work in the copy so your answers are saved.
2. Run the cells from top to bottom (**▶** or **Shift + Enter**); each cell needs the cells above it.
3. Where you see **Your answer**, double-click the cell and write your answer **before** you run the next code cell. A guess written down first is how you find out what you really expected.
4. Fill each `____`, run the cell, and compare with the **Success check** below it. Your last digit or two may differ: a difference in the third decimal place is fine.
5. Stuck on a blank? Open the folded Hint. The folded Answers are on the [worksheet](https://ml.wxing.me/materials/lesson9/lab.html); if you fall behind, copy one and carry on.
6. The folded **Try with AI** boxes are for when you finish early, or after class.

Keras is already installed in Colab; you do not need a GPU. Each training cell says roughly how long it runs.

# Part 0: Load MNIST and split it (5 minutes)

**Goal:** the same images and the same split as Assignment 2.

## 0.1 Load and split

Run the cell without changing it. It downloads about 11 MB the first time.

In [ ]:
import numpy as np                                        # numbers and arrays
import matplotlib.pyplot as plt                           # plots
from tensorflow import keras                              # neural networks

(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()   # 60,000 training images, 10,000 test images
X = x_train.reshape(-1, 784) / 255.0                      # each image becomes 784 pixels between 0 and 1
y = y_train                                               # the digit, 0 to 9
X_fit = X[:50000]                                         # the first 50,000 images: fit the models
y_fit = y[:50000]                                         # and their digits
X_val = X[50000:]                                         # the last 10,000 images: compare the models
y_val = y[50000:]                                         # and their digits
print(X_fit.shape, X_val.shape)                           # images, pixels per image
print(y_fit[:10])                                         # the first ten labels

**Success check:** `(50000, 784) (10000, 784)`, then `[5 0 4 1 9 2 1 3 1 4]`.

We do not use `x_test` today. As in Assignment 2, the test images are used once, at the very end.

# Part 1: Lines around the ring (13 minutes)

**Goal:** train the small network of Sections 2 to 4 of the Note yourself, and see how many hidden units it takes to enclose the inner ring.

The data are the ring of Lesson 4: 150 inner points (class 1) and 150 outer points (class 0), two coordinates each. In Section 2 of the Note you placed three lines by hand. Here Keras places them.

## 1.1 One unit, two, three, four

Each hidden unit is one small logistic regression (Section 2 of the Note): it computes a weighted sum $z = w_1x_1 + w_2x_2 + b$ and squeezes it into a number between 0 and 1. The set $z = 0$ is a straight line. The output unit on top is one more logistic regression.

**Question first:** a network with only **1** hidden unit has one line. What shape can the region it calls "inner ring" be?

Fill the blank with the function each hidden unit uses to squeeze its weighted sum, then run. The cell trains four networks, with 1, 2, 3 and 4 hidden units, 300 steps each: about half a minute in Colab.

**Your answer:** *(double-click to write)*

In [ ]:
rng = np.random.default_rng(0)                            # the random numbers of the Lesson 4 ring
angle = rng.uniform(0, 2 * np.pi, 300)                    # an angle for every point
inner_r = rng.normal(1.0, 0.25, 150)                      # radius of the 150 inner points
outer_r = rng.normal(2.4, 0.3, 150)                       # radius of the 150 outer points
radius = np.concatenate([inner_r, outer_r])               # all 300 radii
x1 = radius * np.cos(angle)                               # first coordinate
x2 = radius * np.sin(angle)                               # second coordinate
X_ring = np.column_stack([x1, x2])                        # 300 rows, two columns
y_ring = np.repeat([1, 0], 150)                           # 1 = inner ring (first 150), 0 = outer ring

ring_models = {}                                          # the trained networks, by number of hidden units
for units in [1, 2, 3, 4]:                                # 1, 2, 3 and 4 hidden units
    keras.utils.set_random_seed(0)                        # the same random start for every network
    ring_model = keras.Sequential([
        keras.layers.Input((2,)),                         # two coordinates in
        keras.layers.Dense(units, activation=____),       # the hidden units: small logistic regressions
        keras.layers.Dense(1, activation="sigmoid"),      # one more logistic unit on top: probability of inner ring
    ])
    ring_model.compile(optimizer=keras.optimizers.Adam(0.05),   # a larger step size than the default
                       loss="binary_crossentropy",              # cross-entropy for two classes
                       metrics=["accuracy"])                    # also report accuracy
    hist_ring = ring_model.fit(X_ring, y_ring, epochs=300,      # 300 steps
                               batch_size=300, verbose=0)       # all 300 points in every step
    ring_models[units] = ring_model                           # keep the network for the plots in 1.2
    print("hidden units", units, "accuracy", round(hist_ring.history["accuracy"][-1], 3))   # training accuracy

<details><summary><b>Hint</b> (only if you are stuck)</summary>

the same function as the output unit: each hidden unit is a logistic regression.

</details>

**Success check:**

```
hidden units 1 accuracy 0.69
hidden units 2 accuracy 0.82
hidden units 3 accuracy 0.97
hidden units 4 accuracy 0.997
```

These are the numbers of the widget in Section 3 of the Note: the same data, the same network and the same training.

## 1.2 Draw the lines

Run the cell without changing it. It shades the region the network calls inner ring (probability above 0.5) and draws one green line per hidden unit, where that unit's weighted sum $z$ is 0. Left: the network with 1 unit; right: the network with 4 units.

In [ ]:
ticks = np.linspace(-3.6, 3.6, 200)                       # 200 positions along each axis
gx, gy = np.meshgrid(ticks, ticks)                        # a 200 x 200 grid of points
grid = np.column_stack([gx.ravel(), gy.ravel()])          # the grid as 40,000 rows of two coordinates
colours = np.where(y_ring == 1, "#0072B2", "#E69F00")     # inner ring blue, outer ring orange
hidden = [1, 4]                                           # left: 1 hidden unit; right: 4
fig, axes = plt.subplots(1, 2, figsize=(10, 5))           # two plots side by side
for i in [0, 1]:                                          # the left plot, then the right plot
    units = hidden[i]                                     # the number of hidden units
    net = ring_models[units]                              # the network trained in 1.1
    prob = net.predict(grid, verbose=0)                   # probability of inner ring at every grid point
    prob = prob.reshape(gx.shape)                         # back to the 200 x 200 grid
    W1, b1 = net.layers[0].get_weights()                  # the weights and constants of the hidden units
    Z = grid @ W1 + b1                                    # each hidden unit's weighted sum z at every grid point
    axes[i].contourf(gx, gy, prob > 0.5, levels=[0.5, 1.5], colors=["#cfe3f3"])   # shade: called inner ring
    for k in range(units):                                # one green line per hidden unit: where its z is 0
        axes[i].contour(gx, gy, Z[:, k].reshape(gx.shape), levels=[0], colors="#00795A")
    axes[i].scatter(X_ring[:, 0], X_ring[:, 1], c=colours, s=12)   # the 300 points
    axes[i].set_title("hidden units: " + str(units))
    axes[i].set_aspect("equal")
plt.show()

**Success check:** two plots of the 300 points with a shaded region, 1 green line on the left and 4 green lines on the right. On the left, the edge of the shading is parallel to the green line but need not lie on it.

**Questions:**

1. On the right, what do the 4 green lines enclose? Each hidden unit only says which side of its own line a point is on. Which layer turns the 4 lines into "inside all four lines"? Answer in one sentence; "the output unit" and "the second layer" are both right names.
2. Put `"linear"` in the blank of 1.1 instead (no squeeze: each unit passes on its weighted sum), and run 1.1 and 1.2 again (about half a minute). What is the accuracy with 4 units, and what is the edge of the shaded region now? Then put `"sigmoid"` back in the blank; you need not run it again: the ring networks are not used again after Part 1, so this does not affect Part 2.

**Your answer:** *(double-click to write)*

<details><summary><b>If you are curious: why 0.69 with one line, and not the 0.50 of Lesson 4?</b></summary>

Both are one line; what differs is where it lies. The logistic regression of Lesson 4 lets its probability grow steadily across the whole plane, and the best such line on a ring runs through the middle: 0.50. The hidden unit squeezes its sum, so points far inside its side all count about the same. Training could then move the line out to the edge, where it cuts off a slice of the outer ring (the white region on the left) that holds only orange points: those points are right, and the rest of the plane is called inner ring, which gives 0.69.

</details>

# Part 2: Build and train the network (14 minutes)

**Goal:** the network of Assignment 2, Section 4: 784 pixels in, two hidden layers of 128 units, 10 outputs.

## 2.1 Build the network

**Question first:** the output layer must give a probability for each of the 10 digits. Which function from Lesson 4 does that?

Fill the blank: the output layer uses the function from your answer. The first hidden layer is given: 128 ReLU units, like the second.

**Your answer:** *(double-click to write)*

In [ ]:
keras.utils.set_random_seed(0)                            # fix the random starting weights and batches
model = keras.Sequential([
    keras.layers.Input((784,)),                           # 784 pixels in
    keras.layers.Dense(128, activation="relu"),            # hidden layer 1
    keras.layers.Dense(128, activation="relu"),           # hidden layer 2: 128 ReLU units
    keras.layers.Dense(10, activation=____),              # output: one probability per digit
])
print(model.count_params())                               # how many weights the network has

<details><summary><b>Hint</b> (only if you are stuck)</summary>

the activation names are text, in quotation marks, for example `"relu"`.

</details>

**Success check:** `118282`.

## 2.2 Where do 118,282 weights come from?

Each unit has one weight for every input, plus one constant. Run the cell without changing it: it counts the weights layer by layer.

In [ ]:
layer1 = 784 * 128 + 128                                  # 128 units, each with 784 weights and one constant
layer2 = 128 * 128 + 128                                  # 128 units, each with 128 weights and one constant
output = 128 * 10 + 10                                    # 10 units, each with 128 weights and one constant
print(layer1, layer2, output, layer1 + layer2 + output)   # the three layers and the total
print(784 * 10 + 10)                                      # logistic regression on the same pixels (Lesson 4)

**Success check:** `100480 16512 1290 118282`, then `7850`.

**Question:** the network has how many times as many weights as logistic regression? What risk does that bring (Lesson 6)?

**Your answer:** *(double-click to write)*

## 2.3 Train for 5 epochs

**Question first:** in the lecture, one epoch with batches of 32 images was 1563 steps. Roughly how many steps are 5 epochs?

Run the cell without changing it. It takes about half a minute in Colab.

**Your answer:** *(double-click to write)*

In [ ]:
model.compile(optimizer="adam",                           # gradient descent that sets its own step sizes
              loss="sparse_categorical_crossentropy",     # cross-entropy; the labels are integers 0 to 9
              metrics=["accuracy"])                       # also report accuracy
hist = model.fit(X_fit, y_fit, epochs=5,                  # 5 passes over the 50,000 fitting images
                 validation_data=(X_val, y_val),          # loss and accuracy on the validation set after every epoch
                 verbose=0)                               # print nothing while training
print(np.round(hist.history["val_accuracy"], 4))          # validation accuracy after each epoch

**Success check:** five numbers between about `0.96` and `0.975`; the last is about `0.970`.

If you run this cell again, run 2.1 first: otherwise `fit` carries on from the weights it has already trained, and the numbers will not match.

**Question:** logistic regression on the same split gives a validation accuracy of `0.9277` (Assignment 2, Table 1, asks you to compute it). Out of the 10,000 validation images, how many fewer mistakes does the network make?

**Your answer:** *(double-click to write)*

<details><summary><b>Try with AI (if you finish early, or after class)</b></summary>

**Using AI: the four steps**

The **Try with AI** boxes use an AI assistant for the Python you do not know yet, while you stay in charge of the machine learning. Every box has the same four steps:

1. **You want:** say what you want in machine-learning words (for example, "the weights of each first-layer unit as a 28 × 28 image").
2. **Tell the AI:** the names of the variables it should use.
3. **Ask:** the request.
4. **Check:** compare the AI's result with a number or shape given on this page. If it does not match, the AI or your question is wrong. Find out which.

---

**You want:** to see what the units of the first hidden layer look for.

**Tell the AI:** `model.layers[0].get_weights()[0]` is a 784 × 128 array: one column of 784 weights per unit.

**Ask:** *"Take the first 10 columns of model.layers[0].get_weights()[0], reshape each to 28 × 28, and show them in one row with the colour map 'RdBu_r', centred at 0, with the title 'unit k'."*

**Check:** 10 red and blue patterns. None of them looks like a whole digit; most respond to a few areas of the image (compare Section 8 of the Note).

</details>

# Part 3: Remove the activation in the middle (5 minutes)

**Goal:** see what the network loses without a bend in the middle.

## 3.1 No activation

**Question first:** on the ring (1.2, Question 2), removing the activation left one straight line at 0.503, however many hidden units. What will it do on MNIST: near 0.97, near logistic regression's 0.9277, or much lower?

Run the cell without changing it: both hidden layers use `"linear"` (no activation: each unit passes on its weighted sum). It takes about half a minute.

**Your answer:** *(double-click to write)*

In [ ]:
keras.utils.set_random_seed(0)                            # the same starting point as Part 2
flat = keras.Sequential([
    keras.layers.Input((784,)),                           # 784 pixels in
    keras.layers.Dense(128, activation="linear"),             # hidden layer 1, no bend
    keras.layers.Dense(128, activation="linear"),             # hidden layer 2, no bend
    keras.layers.Dense(10, activation="softmax"),         # output as before
])
flat.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])   # as in 2.3
hist_flat = flat.fit(X_fit, y_fit, epochs=5, validation_data=(X_val, y_val), verbose=0)       # as in 2.3
print(np.round(hist_flat.history["val_accuracy"], 4))     # validation accuracy after each epoch

<details><summary><b>Success check</b> (open after you run the cell)</summary>

five numbers around `0.92`; the last is about `0.920`.

</details>

**Question:** this network still has 118,282 weights. Why is it no better than logistic regression?

**Your answer:** *(double-click to write)*

# Part 4: How many epochs? (8 minutes)

**Goal:** the training and validation curves, as in Figure 2 of Assignment 2.

## 4.1 Train for 8 epochs and draw the curves

**Question first:** the training loss falls at every epoch. Will the validation loss do the same?

Run the cell without changing it. It builds the network of Part 2 again from the same seed and trains it for 8 epochs: about one minute in Colab.

**Your answer:** *(double-click to write)*

In [ ]:
keras.utils.set_random_seed(0)                            # start again from the same point
model = keras.Sequential([
    keras.layers.Input((784,)),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dense(10, activation="softmax"),
])                                                        # the network of Part 2
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])   # as in 2.3
hist = model.fit(X_fit, y_fit, epochs=8, validation_data=(X_val, y_val), verbose=0)   # 8 epochs
h = hist.history                                          # four lists: loss, val_loss, accuracy, val_accuracy
epochs = np.arange(1, 9)                                  # 1 to 8

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))         # two plots side by side
axes[0].plot(epochs, h["loss"], marker="o", label="training")         # left: the losses
axes[0].plot(epochs, h["val_loss"], marker="o", label="validation")
axes[0].set_xlabel("epoch")                               # labels and legend of the left plot
axes[0].set_ylabel("loss")
axes[0].legend()
axes[1].plot(epochs, h["accuracy"], marker="o", label="training")     # right: the accuracies
axes[1].plot(epochs, h["val_accuracy"], marker="o", label="validation")
axes[1].set_xlabel("epoch")                               # labels and legend of the right plot
axes[1].set_ylabel("accuracy")
axes[1].legend()
plt.show()                                                # draw both plots
print(np.argmin(h["val_loss"]) + 1, round(min(h["val_loss"]), 3))     # the epoch with the lowest validation loss, and its value

<details><summary><b>Success check</b> (open after you run the cell)</summary>

two plots. The lowest validation loss is at epoch `4`, about `0.093`.

</details>

**Questions:**

1. After epoch 4, what do the training loss and the validation loss each do?
2. In one sentence: after epoch 4, is the network learning more about digits, or memorising the training images? What is your evidence? (Look at the validation **loss**; the validation accuracy may still creep up a little.)

**Your answer:** *(double-click to write)*

<details><summary><b>Try with AI (if you finish early, or after class)</b></summary>

**You want:** training that stops by itself when the validation loss stops improving.

**Tell the AI:** the model is built and compiled as in 4.1; `X_fit`, `y_fit`, `X_val`, `y_val` as above.

**Ask:** *"Rebuild the model of 4.1 with keras.utils.set_random_seed(0), then train it for at most 20 epochs with keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True). Print the number of epochs it ran and the final validation accuracy."*

**Check:** it stops at epoch 7 (three epochs after the best, epoch 4) and keeps the weights of epoch 4; the validation accuracy is about `0.974`.

</details>

# Part 5: Does the seed matter? (5 minutes)

**Goal:** see how much the result moves when only the random seed changes. To save time in class, this part has no training: read the results below.

**Question first:** the seed fixes the starting weights and the order of the batches. If you change it, will the validation accuracy after 5 epochs still be exactly 0.970?

**Your answer:** *(double-click to write)*

<details><summary><b>The results: open after you have written your answer</b></summary>

The network of Part 2 was trained twice more, changing only the seed:

| | seed 0 (Part 2) | seed 1 |
|---|---:|---:|
| validation accuracy after 5 epochs | 0.9700 | 0.9760 |

After 20 epochs (Section 7 of the Note), seeds 0, 1 and 2 give 0.9780, 0.9755 and 0.9781. Use the 20-epoch spread, 0.26 percentage points, as the size of the seed's luck, not the 0.6 points at 5 epochs: after 20 epochs the networks have settled, while at 5 epochs seed 0 happens to be in a dip.

</details>

**Question:** (If you are short of time, answer this one after class.) A classmate's changed network beats yours by 0.1 percentage points. Is that evidence that their network is better? Use the 20-epoch spread (0.26 points) in your answer. What would you ask them to do?

**Your answer:** *(double-click to write)*

**After class: run it yourself (about half a minute in Colab)**

In [ ]:
keras.utils.set_random_seed(1)                            # a different seed; nothing else changes
other = keras.Sequential([
    keras.layers.Input((784,)),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dense(10, activation="softmax"),
])                                                        # the network of Part 2
other.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])   # as in 2.3
hist_other = other.fit(X_fit, y_fit, epochs=5, validation_data=(X_val, y_val), verbose=0)       # 5 epochs, as in 2.3
print(round(hist_other.history["val_accuracy"][-1], 4))   # validation accuracy after 5 epochs: about 0.976

<details><summary><b>Try with AI (if you finish early, or after class)</b></summary>

**You want:** which digits the network confuses, as a 10 × 10 confusion matrix on the validation set (Lesson 4).

**Tell the AI:** `model` is the network trained in 4.1; `X_val` and `y_val` are the validation images and labels.

**Ask:** *"Compute model.predict(X_val), take the argmax of each row as the predicted digit, and print sklearn.metrics.confusion_matrix(y_val, predicted). Then print the three largest numbers off the diagonal with their true and predicted digits."*

**Check:** the diagonal adds up to about `9,740` of the 10,000 images (the validation accuracy of 4.1 after 8 epochs). Each off-diagonal number is small (a few tens at most).

</details>

# After class: how today's lab helps with Assignment 2

Assignment 2 (handwritten digits, MNIST) asks for this network in Section 4. **From today you can do the training and Figure 2.**

| What you did today | Where it appears in Assignment 2 |
|---|---|
| Part 0: load MNIST, first 50,000 to fit, last 10,000 to validate | Data: "A fixed split, used throughout". The assignment calls the arrays `X_train`, `y_train` (our `X_fit`, `y_fit`) and `X_val`, `y_val`. Its Option 1 loads the same Keras file, joins the training and test images, then splits them; the first 50,000 and the next 10,000 are the same images as today |
| Part 2: `Sequential`, `Dense(128, "relu")` twice, `Dense(10, "softmax")`, `compile`, `fit(validation_data=...)` | Section 4: "two hidden layers of 128 ReLU units each … the Adam optimiser and the loss `sparse_categorical_crossentropy` … pass the validation set as `validation_data`" |
| Part 4: the two plots of loss and accuracy against the epoch | Section 4, **Figure 2**: "on the left, the training loss and the validation loss … on the right, the training accuracy and the validation accuracy" |
| Part 5: `set_random_seed`, and how much one seed moves the result | Randomness: "write `keras.utils.set_random_seed(0)` before you build the model"; Going further: "Train again with three different random seeds" |
| The Try with AI box after Part 5: the confusion matrix | Section 4: Table 5, "the 5 most frequent kinds of mistake" (in the assignment, on `test`, once) |

**Two things that are different in the assignment.**

1. The assignment trains for **20 epochs**, not 8. It takes about two to three minutes in Colab.
2. In the assignment, the final model is checked on `test` **once**, after you have chosen it on the validation set. Today we never touched `x_test`.

**Reflection** (after class, one or two sentences each):

1. What can 4 hidden units do on the ring that 1 unit cannot, and why must there be an activation between the layers?
2. What does one epoch mean, and how many steps is it with batches of 32 on 50,000 images?
3. Why do we decide the number of epochs from the validation curve and not from the training curve?
4. How large a difference between two networks would you trust, and why?

**Your answer:** *(double-click to write)*

# Advanced Study (after class, not part of the 50 minutes)

MPS311 students are encouraged to try one. **MPS439 students complete A**, the main task, and record it in the log at the end. Allow about 30 minutes. B and C are extras for everyone.

## A. A one-hidden-layer network in numpy, with ReLU

Advanced Study 2 of the Note trains this network with sigmoid hidden units. Here you use ReLU, so its numbers will not match yours.

**Tasks**

1. Build the ring of Lesson 4 as `X_ring` (300 × 2) and `y_ring` (the first lines of 1.1 do this; so does Section 9 of the Note).
2. Using only numpy, train a network with 4 **ReLU** hidden units and one logistic output by gradient descent: 5000 steps, step size 0.5, all 300 points in every step. Start from a new `rng = np.random.default_rng(0)` (make it again after building the ring): first `W1 = rng.normal(0, 1, (2, 4))`, then `w2 = rng.normal(0, 1, 4)`; both constants start at 0. Write the gradients yourself with the four steps of Advanced Study 1 of the Note; only step 3 changes, because the slope of ReLU is 1 where its weighted sum is positive and 0 elsewhere.
3. Print the loss at the start and at the end, and the training accuracy. Compare with Keras (1.1: 0.997 with 4 sigmoid units).

**Ask AI**

Ask: *"Why does the gradient of the cross-entropy with respect to the output score of a logistic unit equal p − y?"* Check its argument against Lesson 4, Advanced Study 1.

<details><summary><b>Hint</b> (only if you are stuck)</summary>

For sigmoid units the four steps in numpy are `g = (p - y_ring) / 300`; `gw2 = H.T @ g`; `gH = np.outer(g, w2) * H * (1 - H)`; `gW1 = X_ring.T @ gH`. For ReLU, keep the weighted sums `Zh = X_ring @ W1 + b1`, use `H = np.maximum(0, Zh)`, and replace `H * (1 - H)` by `(Zh > 0)`. Each step updates with the step size, for example `W1 = W1 - 0.5 * gW1` (the same for `b1`, `w2`, `b2`). The loss is the mean cross-entropy, `-np.mean(y_ring * np.log(p) + (1 - y_ring) * np.log(1 - p))`; compute it before the first step and after the last. The constants take the sums `g.sum()` and `gH.sum(axis=0)`.

</details>

In [ ]:
# Your code here: add more cells with + Code if you need them

**Self-check:** the loss falls from about `0.651` to about `0.019`; the accuracy is `0.997`.

## B. Check one gradient by hand

After A, pick one weight, for example `W1[0, 0]`. Compute the loss with `1e-5` added to it and with `1e-5` taken away, and divide the difference by `2e-5`. Compare with `gW1[0, 0]` computed again at the final weights (the last `gW1` inside your loop used the weights before the last update). (Advanced Study 2 of the Note adds `1e-5` only once; stepping both ways, a central difference, is more accurate, and here it is needed to agree to three figures.)

In [ ]:
# Your code here: add more cells with + Code if you need them

**Self-check:** the two numbers agree in their first three significant figures (about `0.0000575`). Then explain in one sentence why this check is too slow to train the network of Part 2 with (Section 5 of the Note).

## C. Dropout

Rebuild the network of 4.1 with `keras.layers.Dropout(0.2)` after each hidden layer, train for 8 epochs, and draw the same two plots.

In [ ]:
# Your code here: add more cells with + Code if you need them

**Self-check:** the lowest validation loss is now at epoch `7` (about `0.078`), and after 8 epochs the validation loss is only about `0.016` above the training loss, against about `0.091` in 4.1.

## Advanced Study log

Double-click the cell below and fill it in.

> **Investigations done (A, B or C):**
>
> **What I found:**
>
> **How AI helped, and what I had to correct:**
>
> **Self-check result (passed or not):**

## 30-second feedback

Share anonymous feedback or post a question for this lab on the [course's Class page](https://ml.wxing.me/live/) (Give feedback, then choose Lesson 9 lab).